In [2]:
ls /media/notebooks/dataset/ 

TABLA_ACCIDENTES_24.avro  TABLA_ACCIDENTES_24.csv  TABLA_ACCIDENTES_24.parquet


In [23]:
!hdfs dfs -mkdir -p /datasets/accidentes/resultados/dgt_accidentes_provincia_mes
!hdfs dfs -mkdir -p /datasets/accidentes/csv
!hdfs dfs -put  /media/notebooks/dataset/TABLA_ACCIDENTES_24.csv /datasets/accidentes/csv

!hdfs dfs -ls /datasets/accidentes/csv

put: `/datasets/accidentes/csv/TABLA_ACCIDENTES_24.csv': File exists
Found 1 items
-rw-r--r--   3 root supergroup   19043901 2026-10-23 09:35 /datasets/accidentes/csv/TABLA_ACCIDENTES_24.csv


In [24]:
%%writefile mapper_accidentes.py

#!/usr/bin/env python3
import sys
import csv

# Índices de las columnas del CSV
IDX_MES = 2
IDX_PROVINCIA = 5

# Correspondencia entre códigos y nombres de provincias
PROVINCIAS = {
    "01": "Araba/Álava",
    "02": "Albacete",
    "03": "Alicante/Alacant",
    "04": "Almería",
    "05": "Ávila",
    "06": "Badajoz",
    "07": "Illes Balears",
    "08": "Barcelona",
    "09": "Burgos",
    "10": "Cáceres",
    "11": "Cádiz",
    "12": "Castellón/Castelló",
    "13": "Ciudad Real",
    "14": "Córdoba",
    "15": "A Coruña",
    "16": "Cuenca",
    "17": "Girona",
    "18": "Granada",
    "19": "Guadalajara",
    "20": "Gipuzkoa",
    "21": "Huelva",
    "22": "Huesca",
    "23": "Jaén",
    "24": "León",
    "25": "Lleida",
    "26": "La Rioja",
    "27": "Lugo",
    "28": "Madrid",
    "29": "Málaga",
    "30": "Murcia",
    "31": "Navarra",
    "32": "Ourense",
    "33": "Asturias",
    "34": "Palencia",
    "35": "Las Palmas",
    "36": "Pontevedra",
    "37": "Salamanca",
    "38": "Santa Cruz de Tenerife",
    "39": "Cantabria",
    "40": "Segovia",
    "41": "Sevilla",
    "42": "Soria",
    "43": "Tarragona",
    "44": "Teruel",
    "45": "Toledo",
    "46": "Valencia/València",
    "47": "Valladolid",
    "48": "Bizkaia",
    "49": "Zamora",
    "50": "Zaragoza",
    "51": "Ceuta",
    "52": "Melilla"
}

for linea in sys.stdin:
    try:
        fila = next(csv.reader([linea], delimiter=";"))

        # Omitir la cabecera
        if not fila or fila[0].lstrip("\ufeff") == "ID_ACCIDENTE":
            continue

        codigo_provincia = fila[IDX_PROVINCIA].strip().zfill(2)
        mes = fila[IDX_MES].strip()

        if not codigo_provincia or not mes:
            continue

        # Obtener el nombre de la provincia
        provincia = PROVINCIAS.get(
            codigo_provincia,
            f"Provincia desconocida ({codigo_provincia})"
        )

        mes = str(int(mes))

        # Clave: nombre de provincia y mes
        print(f"{provincia}|{mes}\t1")

    except (IndexError, ValueError, csv.Error):
        continue


Overwriting mapper_accidentes.py


In [12]:
%%writefile reducer_accidentes.py

#!/usr/bin/env python3
import sys

clave_anterior = None
total = 0

for linea in sys.stdin:
    linea = linea.strip()

    if not linea:
        continue

    clave, valor = linea.split("\t", 1)
    valor = int(valor)

    if clave == clave_anterior:
        total += valor
    else:
        if clave_anterior is not None:
            provincia, mes = clave_anterior.split("|")
            print(f"{provincia}\t{mes}\t{total}")

        clave_anterior = clave
        total = valor

if clave_anterior is not None:
    provincia, mes = clave_anterior.split("|")
    print(f"{provincia}\t{mes}\t{total}")


Overwriting reducer_accidentes.py


In [16]:
!chmod +x mapper_accidentes.py reducer_accidentes.py
!hdfs dfs -ls /datasets/accidentes/

Found 2 items
drwxr-xr-x   - root supergroup          0 2026-10-23 09:35 /datasets/accidentes/csv
drwxr-xr-x   - root supergroup          0 2026-10-23 09:34 /datasets/accidentes/resultados


In [ ]:
!hdfs dfs -rm -r -f /datasets/accidentes/resultados/dgt_accidentes_provincia_mes

!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files mapper_accidentes.py,reducer_accidentes.py \
    -mapper "python3 mapper_accidentes.py" \
    -reducer "python3 reducer_accidentes.py" \
    -input /datasets/accidentes/csv/TABLA_ACCIDENTES_24.csv \
    -output /datasets/accidentes/resultados/dgt_accidentes_provincia_mes

Deleted /datasets/accidentes/resultados/dgt_accidentes_provincia_mes
packageJobJar: [/tmp/hadoop-unjar155686256109507125/] [] /tmp/streamjob3148653040740437248.jar tmpDir=null
2026-10-23 09:48:47,148 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-10-23 09:48:47,364 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-10-23 09:48:47,668 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1792745247675_0002
2026-10-23 09:48:48,428 INFO mapred.FileInputFormat: Total input files to process : 1
2026-10-23 09:48:48,582 INFO mapreduce.JobSubmitter: number of splits:2
2026-10-23 09:48:48,872 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1792745247675_0002
2026-10-23 09:48:48,872 INFO mapreduce.JobSubmitter: Executing with tokens: []


In [20]:
!hdfs dfs -ls /datasets/accidentes/resultados

Found 2 items
-rw-r--r--   3 root supergroup          0 2026-10-23 09:45 /datasets/accidentes/resultados/_SUCCESS
-rw-r--r--   3 root supergroup       5324 2026-10-23 09:45 /datasets/accidentes/resultados/part-00000


In [21]:
!hdfs dfs -cat /datasets/accidentes/resultados/part-00000


10	1	49
10	10	37
10	11	34
10	12	33
10	2	28
10	3	34
10	4	44
10	5	53
10	6	55
10	7	49
10	8	52
10	9	49
11	1	228
11	10	216
11	11	195
11	12	176
11	2	185
11	3	173
11	4	274
11	5	267
11	6	244
11	7	265
11	8	273
11	9	211
12	1	77
12	10	93
12	11	79
12	12	70
12	2	73
12	3	87
12	4	106
12	5	88
12	6	101
12	7	114
12	8	99
12	9	79
13	1	35
13	10	56
13	11	29
13	12	47
13	2	39
13	3	36
13	4	44
13	5	46
13	6	64
13	7	47
13	8	53
13	9	50
14	1	65
14	10	71
14	11	69
14	12	72
14	2	63
14	3	69
14	4	74
14	5	58
14	6	71
14	7	74
14	8	58
14	9	68
15	1	144
15	10	127
15	11	140
15	12	143
15	2	128
15	3	150
15	4	126
15	5	145
15	6	123
15	7	171
15	8	167
15	9	149
16	1	14
16	10	27
16	11	15
16	12	20
16	2	17
16	3	18
16	4	30
16	5	20
16	6	31
16	7	23
16	8	34
16	9	19
17	1	134
17	10	201
17	11	181
17	12	182
17	2	154
17	3	172
17	4	189
17	5	184
17	6	213
17	7	222
17	8	213
17	9	202
18	1	93
18	10	131
18	11	109
18	12	120
18	2	102
18	3	115
18	4	117
18	5	136
18	6	118
18	7	94
18	8	82
18	9	102
19	1	26
19	10	25
19	11	36
19	12	32
19	2	33
19	3	43
19	4	33
19